# Análise de Clustering: Similaridade Socioeconômica Municipal

**Projeto:** Compliance Público Baseado em Dados (TCC MBA)  
**Título do TCC:** Correlação entre Repasses Federais e Indicadores Socioeconômicos Municipais

**Autor:** Enok  
**Data:** 2026-04-19

---

## Objetivo

Agrupar municípios brasileiros em **clusters de similaridade socioeconômica** utilizando K-Means sobre features normalizadas derivadas do censo do IBGE e dos registros de transferências federais.

A estratégia de clustering permite **comparar laranja com laranja**: municípios com perfis socioeconômicos semelhantes são agrupados juntos, fornecendo a base para a análise de correlação entre corrupção e IDH no Caderno 05.

---

## Hipóteses

1. **H1:** Os municípios brasileiros formam clusters socioeconômicos distintos com base em renda, alfabetização, população e volume de transferências
2. **H2:** A pertença ao cluster é geograficamente concentrada (existem padrões regionais)
3. **H3:** K=4 clusters captura a principal variação estrutural nos dados

---

## Features de Entrada

**Indicadores socioeconômicos (normalizados):**
- `renda_media_real_2022_brl_2022`: Renda média real (IPCA 2022)
- `taxa_alfabetizacao_2022`: Taxa de alfabetização de adultos
- `mudanca_renda_real_pct`: Variação real da renda 2010–2022
- `mudanca_alfabetizacao_pp`: Variação da alfabetização (pp) 2010–2022
- `log_populacao`: Populacao log-transformada (2022)
- `log_total_transferencias`: Transferências federais log-transformadas

**Redução de dimensionalidade:**
- PCA (3 componentes) antes do K-Means

---

## Saídas Esperadas

1. **Atribuição de clusters:** Rótulos de cluster por município (K=4)
2. **Coordenadas PCA:** PC1, PC2, PC3 para cada município
3. **Perfis de cluster:** Estatísticas resumidas por cluster
4. **Scatter PCA interativo:** Municípios no espaço PCA coloridos por cluster
5. **Dataset de clustering consolidado:** Salvo na camada Gold para análise downstream

In [ ]:
# --- AUTO-GENERATED DEPENDENCY INSTALL ---
# Installs all project dependencies on first run (Colab, fresh environments, etc).
# Idempotent: pip skips anything already installed.
# To regenerate this cell, run: python scripts/inject_pip_install.py

import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Installing dependencies from {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependencies ready.")
else:
    print("requirements.txt not found. Install manually: pip install -r requirements.txt")


## Passo a passo

1. Pacotes e configuração do ambiente
2. Reprodutibilidade
3. Carregamento de dados
4. Blocos de análise
5. Resumo e interpretação


## 1. Configuração e Carregamento dos Dados

# Pacotes


In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

# AWS
import boto3
import tempfile

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Libraries loaded successfully!")


# Reprodutibilidade


In [ ]:
import os
import random

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"Seed de reprodutibilidade fixada em {SEED}")


In [ ]:
import json as _json

_rtcfg_path = os.path.join('..', 'config', 'runtime_config.json')
if os.path.exists(_rtcfg_path):
    with open(_rtcfg_path) as _f:
        _rtcfg = _json.load(_f)
else:
    _rtcfg = {}

S3_BUCKET_NAME = os.environ.get('S3_BUCKET_NAME', _rtcfg.get('aws', {}).get('s3_bucket_name', ''))
AWS_PROFILE = os.environ.get('AWS_PROFILE', _rtcfg.get('aws', {}).get('profile', None))


In [ ]:
# Carrega os dados da camada Gold local
from src.analysis.pt_br_loader import GoldDataLoaderPtBr as LocalGoldDataLoader

loader = LocalGoldDataLoader()
df = loader.load_dataset('clustering_consolidado')

if df is None:
    raise FileNotFoundError("Dataset consolidated_clustering não encontrado em data/gold/")

print(f"Carregadas {len(df)} linhas da camada Gold local")
print(f"Colunas: {list(df.columns)}")


In [ ]:
# Exibe as primeiras linhas
df.head()


In [ ]:
# Verifica a qualidade dos dados
print("=" * 60)
print("VERIFICAÇÃO DE QUALIDADE DOS DADOS")
print("=" * 60)
print(f"\nTotal de municípios: {len(df):,}")
print(f"Municípios únicos: {df['codigo_municipio'].nunique():,}")
print(f"Municípios duplicados: {len(df) - df['codigo_municipio'].nunique()}")
print(f"\nValores ausentes por coluna:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values!")


## 2. Estatísticas Descritivas

In [ ]:
# Define as colunas de features brutas (não-normalizadas)
raw_features = [
    'populacao_2010', 'populacao_2022', 'mudanca_populacao_pct',
    'taxa_alfabetizacao_2010', 'taxa_alfabetizacao_2022', 'mudanca_alfabetizacao_pp',
    'renda_media_real_2010_brl_2022', 'renda_media_real_2022_brl_2022', 'mudanca_renda_real_pct',
    'domicilios_2010', 'domicilios_2022', 'mudanca_domicilios_pct'
]

# Estatísticas descritivasistics
print("=" * 80)
print("DESCRIPTIVE STATISTICS - Raw Features")
print("=" * 80)
df[raw_features].describe().round(2).T


In [ ]:
# Distribuição por região
print("\n" + "=" * 60)
print("DISTRIBUIÇÃO POR REGIÃO")
print("=" * 60)

region_stats = df.groupby('nome_regiao').agg({
    'codigo_municipio': 'count',
    'populacao_2022': ['sum', 'mean', 'median'],
    'taxa_alfabetizacao_2022': 'mean',
    'renda_media_real_2022_brl_2022': 'mean'
}).round(2)

region_stats.columns = ['N_Municipios', 'Pop_Total', 'Pop_Media', 'Mediana_Pop', 
                        'Mean_Literacy', 'Mean_Income']
region_stats = region_stats.sort_values('N_Municipios', ascending=False)
region_stats


In [ ]:
# Distribuição por estado
print("\n" + "=" * 60)
print("TOP 10 ESTADOS POR NÚMERO DE MUNICÍPIOS")
print("=" * 60)

state_counts = df.groupby(['nome_estado', 'nome_regiao']).size().reset_index(name='num_municipios')
state_counts = state_counts.sort_values('num_municipios', ascending=False).head(10)
state_counts


In [ ]:
# Visualiza distribuições das features principais
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# População 2022 (escala log)
ax = axes[0, 0]
ax.hist(np.log10(df['populacao_2022']), bins=50, edgecolor='white', alpha=0.7)
ax.set_xlabel('Log10(População 2022)')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição da População (2022)')

# Taxa de Alfabetização 2022
ax = axes[0, 1]
ax.hist(df['taxa_alfabetizacao_2022'], bins=50, edgecolor='white', alpha=0.7, color='green')
ax.set_xlabel('Literacy Rate (%)')
ax.set_ylabel('Frequência')
ax.set_title('Literacy Rate Distribution (2022)')

# Income 2022
ax = axes[0, 2]
ax.hist(df['renda_media_real_2022_brl_2022'], bins=50, edgecolor='white', alpha=0.7, color='orange')
ax.set_xlabel('Average Income (BRL)')
ax.set_ylabel('Frequência')
ax.set_title('Income Distribution (2022)')

# Population Change
ax = axes[1, 0]
ax.hist(df['mudanca_populacao_pct'], bins=50, edgecolor='white', alpha=0.7, color='purple')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Population Change (%)')
ax.set_ylabel('Frequência')
ax.set_title('Population Change (2010-2022)')

# Literacy Change
ax = axes[1, 1]
ax.hist(df['mudanca_alfabetizacao_pp'], bins=50, edgecolor='white', alpha=0.7, color='teal')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Literacy Change (pp)')
ax.set_ylabel('Frequência')
ax.set_title('Literacy Change (2010-2022)')

# Income Change
ax = axes[1, 2]
ax.hist(df['mudanca_renda_real_pct'], bins=50, edgecolor='white', alpha=0.7, color='brown')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Income Change (%)')
ax.set_ylabel('Frequência')
ax.set_title('Income Change (2010-2022)')

plt.tight_layout()
plt.show()


In [ ]:
# Matriz de correlação
plt.figure(figsize=(14, 10))
corr_matrix = df[raw_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Matriz de Correlação — Features Socioeconômicas', fontsize=14)
plt.tight_layout()
plt.show()


## 3. PCA — Análise de Componentes Principais

PCA nos permite:
1. Reduzir a dimensionalidade preservando a variância
2. Identificar quais features contribuem mais para a variância
3. Visualizar municípios em espaço 2D/3D
4. Potencialmente usar menos componentes para o agrupamento

In [ ]:
# Usa features normalizadas para o PCA
norm_features = [f'{col}_norm' for col in raw_features]

# Extrai os dados normalizados
X_norm = df[norm_features].values

print(f"Shape da matriz de features: {X_norm.shape}")
print(f"Número de features: {len(norm_features)}")


In [ ]:
# Executa PCA com todos os componentes
pca_full = PCA()
pca_full.fit(X_norm)

# Variância explicada
explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# Exibe a variância explicada
print("=" * 60)
print("PCA — VARIÂNCIA EXPLICADA")
print("=" * 60)
pca_df = pd.DataFrame({
    'Component': [f'PC{i+1}' for i in range(len(explained_var))],
    'Variancia_Explicada': explained_var * 100,
    'Cumulative_Variance': cumulative_var * 100
})
print(pca_df.round(2).to_string(index=False))


In [ ]:
# Scree plot e variância acumulada
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
ax = axes[0]
components = range(1, len(explained_var) + 1)
ax.bar(components, explained_var * 100, alpha=0.7, label='Individual')
ax.plot(components, cumulative_var * 100, 'ro-', label='Acumulada')
ax.axhline(y=80, color='green', linestyle='--', label='limiar de 80%')
ax.set_xlabel('Componente Principent')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('PCA Scree Plot')
ax.legend()
ax.set_xticks(components)

# Cumulative variance
ax = axes[1]
ax.plot(components, cumulative_var * 100, 'b-o', linewidth=2, markersize=8)
ax.axhline(y=80, color='green', linestyle='--', label='limiar de 80%')
ax.axhline(y=90, color='orange', linestyle='--', label='90% threshold')
ax.axhline(y=95, color='red', linestyle='--', label='95% threshold')
ax.fill_between(components, cumulative_var * 100, alpha=0.3)
ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative Variance Explained (%)')
ax.set_title('Cumulative Variance Explained')
ax.legend()
ax.set_xticks(components)

plt.tight_layout()
plt.show()

# Determine optimal components
n_components_80 = np.argmax(cumulative_var >= 0.80) + 1
n_components_90 = np.argmax(cumulative_var >= 0.90) + 1
print(f"\nComponents needed for 80% variance: {n_components_80}")
print(f"Components needed for 90% variance: {n_components_90}")


In [ ]:
# Cargas do PCA (contribuições das features para cada componente)
loadings = pd.DataFrame(
    pca_full.components_.T,
    columns=[f'PC{i+1}' for i in range(len(explained_var))],
    index=[col.replace('_norm', '') for col in norm_features]
)

print("=" * 60)
print("CARGAS DO PCA (Contribuições das Features)")
print("=" * 60)
print(loadings[['PC1', 'PC2', 'PC3', 'PC4']].round(3))


In [ ]:
# Visualiza o heatmap das cargas
plt.figure(figsize=(12, 8))
sns.heatmap(loadings[['PC1', 'PC2', 'PC3', 'PC4']], annot=True, cmap='RdBu_r', 
            center=0, fmt='.2f', linewidths=0.5)
plt.title('Cargas do PCA — Contribuições das Features para os Componentes Principais', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Transforma os dados para componentes principais
pca_3 = PCA(n_components=3)
X_pca = pca_3.fit_transform(X_norm)

# Adiciona os componentes do PCA ao dataframe
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]
df['PC3'] = X_pca[:, 2]

print(f"Transformação PCA concluída.")
print(f"Variância explicada por 3 componentes: {pca_3.explained_variance_ratio_.sum()*100:.1f}%")


In [ ]:
# Visualização PCA 2D por região
fig = px.scatter(
    df, x='PC1', y='PC2',
    color='nome_regiao',
    hover_data=['nome_municipio', 'nome_estado', 'populacao_2022', 'renda_media_real_2022_brl_2022'],
    title='PCA: Municípios em Espaço 2D (por Região)',
    labels={'PC1': f'PC1 ({pca_3.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'PC2 ({pca_3.explained_variance_ratio_[1]*100:.1f}%)'}
)
fig.update_layout(height=600)
fig.show()


In [ ]:
# Visualização PCA 3D
fig = px.scatter_3d(
    df, x='PC1', y='PC2', z='PC3',
    color='nome_regiao',
    hover_data=['nome_municipio', 'nome_estado'],
    title='PCA: Municípios em Espaço 3D (por Região)',
    labels={'PC1': f'PC1 ({pca_3.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'PC2 ({pca_3.explained_variance_ratio_[1]*100:.1f}%)',
            'PC3': f'PC3 ({pca_3.explained_variance_ratio_[2]*100:.1f}%)'}
)
fig.update_layout(height=700)
fig.show()


## 4. Agrupamento K-means

Usaremos K-means para agrupar municípios baseado em indicadores socioeconômicos:
- População
- Taxas de alfabetização
- Renda
- Indicadores de domicílios

### 4.1 Determinar o Número Ótimo de Clusters

In [ ]:
# Seleciona features para clustering (usando features normalizadas)
clustering_features = [
    'populacao_2022_norm',
    'taxa_alfabetizacao_2022_norm',
    'renda_media_real_2022_brl_2022_norm',
    'domicilios_2022_norm',
    'mudanca_populacao_pct_norm',
    'mudanca_alfabetizacao_pp_norm',
    'mudanca_renda_real_pct_norm',
    'mudanca_domicilios_pct_norm'
]

X_cluster = df[clustering_features].values
print(f"Clustering feature matrix: {X_cluster.shape}")
print(f"Features used: {clustering_features}")


In [ ]:
# Método do Cotovelo e análise de Silhueta
k_range = range(2, 11)
inertias = []
silhouettes = []

print("Avaliando valores de K...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster)
    inertias.append(kmeans.inertia_)
    sil_score = silhouette_score(X_cluster, kmeans.labels_)
    silhouettes.append(sil_score)
    print(f"  K={k}: Inertia={kmeans.inertia_:.0f}, Silhouette={sil_score:.4f}")


In [ ]:
# Gráficos do Cotovelo e da Silhueta
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico do Cotovelo
ax = axes[0]
ax.plot(list(k_range), inertias, 'b-o', linewidth=2, markersize=8)
ax.set_xlabel('Número de Clusters (K)')
ax.set_ylabel('Inércia (soma dos quadrados intra-cluster)')
ax.set_title('Método do Cotovelo')
ax.set_xticks(list(k_range))

# Gráfico de Silhueta
ax = axes[1]
ax.plot(list(k_range), silhouettes, 'g-o', linewidth=2, markersize=8)
ax.set_xlabel('Número de Clusters (K)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Analysis')
ax.set_xticks(list(k_range))

# Highlight best silhouette
best_k = list(k_range)[np.argmax(silhouettes)]
best_sil = max(silhouettes)
ax.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nBest K based on Silhouette Score: {best_k} (score={best_sil:.4f})")


### 4.2 Aplicar K-means com K Ótimo

In [ ]:
# Usa K com base na análise (ajuste se necessário)
OPTIMAL_K = best_k
print(f"Usando K = {OPTIMAL_K} clusters")

# Ajusta o modelo K-means final
kmeans_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
df['cluster'] = kmeans_final.fit_predict(X_cluster)

# Distribuição dos clusters
print("\nDistribuição dos Clusters:")
print(df['cluster'].value_counts().sort_index())


In [ ]:
# Visualiza clusters no espaço PCA (2D)
fig = px.scatter(
    df, x='PC1', y='PC2',
    color='cluster',
    color_continuous_scale='viridis',
    hover_data=['nome_municipio', 'nome_estado', 'nome_regiao', 'populacao_2022', 'renda_media_real_2022_brl_2022'],
    title=f'Clusters K-means (K={OPTIMAL_K}) no Espaço PCA',
    labels={'PC1': f'PC1 ({pca_3.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'PC2 ({pca_3.explained_variance_ratio_[1]*100:.1f}%)',
            'cluster': 'Cluster'}
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(height=600)
fig.show()


In [ ]:
# Visualiza clusters no espaço PCA 3D
fig = px.scatter_3d(
    df, x='PC1', y='PC2', z='PC3',
    color='cluster',
    color_continuous_scale='viridis',
    hover_data=['nome_municipio', 'nome_estado'],
    title=f'Clusters K-means (K={OPTIMAL_K}) no Espaço PCA 3D'
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(height=700)
fig.show()


### 4.3 Perfil dos Clusters

In [ ]:
# Estatísticas dos clusters
print("=" * 80)
print("PERFIS DOS CLUSTERS — Valores Médios")
print("=" * 80)

cluster_stats = df.groupby('cluster').agg({
    'codigo_municipio': 'count',
    'populacao_2022': ['mean', 'median'],
    'taxa_alfabetizacao_2022': 'mean',
    'renda_media_real_2022_brl_2022': 'mean',
    'domicilios_2022': 'mean',
    'mudanca_populacao_pct': 'mean',
    'mudanca_alfabetizacao_pp': 'mean',
    'mudanca_renda_real_pct': 'mean'
}).round(2)

cluster_stats.columns = ['N_Municipalities', 'Mean_Pop', 'Median_Pop', 'Mean_Literacy',
                         'Mean_Income', 'Mean_Households', 'Pop_Change', 'Lit_Change', 'Inc_Change']
cluster_stats


In [ ]:
# Composição dos clusters por região
print("\n" + "=" * 60)
print("COMPOSIÇÃO DOS CLUSTERS POR REGIÃO")
print("=" * 60)

region_cluster = pd.crosstab(df['cluster'], df['nome_regiao'], margins=True)
print(region_cluster)


In [ ]:
# Visualiza composição dos clusters por região
fig = px.histogram(
    df, x='cluster', color='nome_regiao',
    barmode='stack',
    title='Composição dos Clusters por Região',
    labels={'cluster': 'Cluster', 'count': 'Número de Municípios'}
)
fig.update_layout(height=500)
fig.show()


In [ ]:
# Box plots para cada feature por cluster
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

features_to_plot = [
    ('populacao_2022', 'População 2022', True),
    ('taxa_alfabetizacao_2022', 'Taxa de Alfabetização 2022 (%)', False),
    ('renda_media_real_2022_brl_2022', 'Renda Média 2022 (BRL)', False),
    ('domicilios_2022', 'Domicílios 2022', True),
    ('mudanca_populacao_pct', 'Pop Change (%)', False),
    ('mudanca_alfabetizacao_pp', 'Literacy Change (pp)', False),
    ('mudanca_renda_real_pct', 'Income Change (%)', False),
    ('mudanca_domicilios_pct', 'Households Change (%)', False)
]

for ax, (col, title, use_log) in zip(axes, features_to_plot):
    data = np.log10(df[col]) if use_log else df[col]
    ylabel = f'Log10({col})' if use_log else col
    df.boxplot(column=col if not use_log else None, by='cluster', ax=ax)
    if use_log:
        for i, cluster in enumerate(sorted(df['cluster'].unique())):
            cluster_data = np.log10(df[df['cluster'] == cluster][col])
            ax.boxplot(cluster_data, positions=[i+1])
    ax.set_title(title)
    ax.set_xlabel('Cluster')

plt.suptitle('Feature Distributions by Cluster', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Gráfico radar para perfis de cluster
# Normaliza médias dos clusters para comparação
cluster_means = df.groupby('cluster')[raw_features].mean()
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())

# Seleciona features-chave para o radar
radar_features = ['populacao_2022', 'taxa_alfabetizacao_2022', 'renda_media_real_2022_brl_2022', 
                  'domicilios_2022', 'mudanca_populacao_pct', 'mudanca_renda_real_pct']

fig = go.Figure()

for cluster in sorted(df['cluster'].unique()):
    values = cluster_means_norm.loc[cluster, radar_features].values.tolist()
    values.append(values[0])  # Close the radar
    
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=radar_features + [radar_features[0]],
        fill='toself',
        name=f'Cluster {cluster}'
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 1])
    ),
    title='Cluster Profiles (Normalized)',
    height=600
)
fig.show()


### 4.4 Interpretação dos Clusters

In [ ]:
# Gera interpretações dos clusters com base nas estatísticas
print("=" * 80)
print("INTERPRETAÇÃO DOS CLUSTERS")
print("=" * 80)

for cluster in sorted(df['cluster'].unique()):
    cluster_data = df[df['cluster'] == cluster]
    n_muni = len(cluster_data)
    
    print(f"\n--- CLUSTER {cluster} ({n_muni} municípios, {100*n_muni/len(df):.1f}%) ---")
    
    # Population
    pop_mean = cluster_data['populacao_2022'].mean()
    pop_median = cluster_data['populacao_2022'].median()
    pop_size = "Large" if pop_mean > df['populacao_2022'].mean() else "Small"
    print(f"  Population: {pop_size} (mean={pop_mean:,.0f}, median={pop_median:,.0f})")
    
    # Literacy
    lit_mean = cluster_data['taxa_alfabetizacao_2022'].mean()
    lit_level = "High" if lit_mean > df['taxa_alfabetizacao_2022'].mean() else "Low"
    print(f"  Literacy: {lit_level} ({lit_mean:.1f}%)")
    
    # Income
    inc_mean = cluster_data['renda_media_real_2022_brl_2022'].mean()
    inc_level = "High" if inc_mean > df['renda_media_real_2022_brl_2022'].mean() else "Low"
    print(f"  Income: {inc_level} (R$ {inc_mean:,.2f})")
    
    # Growth
    pop_change = cluster_data['mudanca_populacao_pct'].mean()
    growth = "Growing" if pop_change > 0 else "Declining"
    print(f"  Population Trend: {growth} ({pop_change:+.1f}%)")
    
    # Top regions
    top_regions = cluster_data['nome_regiao'].value_counts().head(2)
    print(f"  Main Regions: {', '.join(top_regions.index)}")


In [ ]:
# Exemplos de municípios de cada cluster
print("\n" + "=" * 80)
print("EXEMPLOS DE MUNICÍPIOS DE CADA CLUSTER")
print("=" * 80)

for cluster in sorted(df['cluster'].unique()):
    print(f"\n--- Cluster {cluster} ---")
    examples = df[df['cluster'] == cluster].nlargest(5, 'populacao_2022')[
        ['nome_municipio', 'nome_estado', 'populacao_2022', 'renda_media_real_2022_brl_2022', 'taxa_alfabetizacao_2022']
    ]
    print(examples.to_string(index=False))


## 5. Resumo e Conclusões

### 5.1 Dataset
- **5.565 municípios** analisados (99,9% dos 5.570 totais), com 12 features socioeconômicas cobrindo população, alfabetização, renda (ajustada pela inflação) e domicílios para 2010 e 2022.
- Sem valores faltantes ou duplicados no dataset final de agrupamento.

### 5.2 Resultados do PCA
- **3 componentes principais** explicam **84,4%** da variância total; 4 componentes atingem 92,5%.
- **PC1 (39,2%)**: desenvolvimento socioeconômico geral — carrega em população, renda, alfabetização e domicílios.
- **PC2 (28,8%)**: escala urbana vs desenvolvimento — separa grandes centros populacionais de municípios com alta alfabetização/renda.
- **PC3 (16,5%)**: trajetória de crescimento — carrega em mudança populacional e mudança de domicílios entre 2010 e 2022.

### 5.3 Agrupamento K-Means
- **K ótimo = 4** clusters, selecionado pelo escore de silhueta (0,288).
- **Cluster 0** (821 municípios, 14,8%): municípios grandes, de alta renda, alta alfabetização e em crescimento — grandes centros urbanos (Brasília, Fortaleza, Salvador, Belo Horizonte). Concentrados no Sudeste e Sul.
- **Cluster 1** (2.158 municípios, 38,8%): municípios pequenos, de baixa renda, baixa alfabetização e população estagnada ou em declínio — predominantemente Nordeste e Norte. Representam a lacuna de desenvolvimento.
- **Cluster 2** (2.584 municípios, 46,4%): municípios pequenos a médios, com alta alfabetização e alta renda, mas com população em declínio — principalmente municípios do interior do Sudeste e do Sul.
- **Cluster 3** (2 municípios, 0,04%): megacidades (São Paulo e Rio de Janeiro) — outliers populacionais extremos formando seu próprio cluster.

### 5.4 Principais Insights
- Os municípios brasileiros apresentam uma clara **estrutura dual**: um cluster desenvolvido Sul/Sudeste (Clusters 0, 2, 3) vs um cluster menos desenvolvido Norte/Nordeste (Cluster 1).
- **Renda e alfabetização** são os diferenciadores primários entre os clusters, confirmando a divisão socioeconômica visível no PCA.
- O **declínio populacional** afeta tanto municípios desenvolvidos (Cluster 2) quanto menos desenvolvidos (Cluster 1), mas por razões diferentes — êxodo rural vs estagnação econômica.
- O **escore de silhueta moderado** (0,288) indica fronteiras de cluster sobrepostas, consistente com um gradiente socioeconômico contínuo em vez de grupos nitidamente separados.

### 5.5 Limitações
- O agrupamento usa apenas indicadores derivados do censo — adicionar dados de sanções, fiscais ou de governança poderia revelar agrupamentos mais sutis.
- K-means assume clusters esféricos; métodos baseados em densidade (DBSCAN, HDBSCAN) podem capturar melhor distribuições assimétricas.
- O Cluster 3 com 2 municípios é um artefato de outliers populacionais extremos em vez de um grupo analítico significativo.
- As colunas de renda ajustadas pela inflação usam deflação pelo IPCA para BRL de 2022 — os resultados dependem da série de deflatores usada.


In [ ]:
print("=" * 80)
print("RESUMO DA ANÁLISE")
print("=" * 80)

print(f"""
DATASET:
  - Total municipalities analyzed: {len(df):,}
  - Features used: {len(raw_features)}
  - No missing values or duplicates

RESULTADOS DO PCA:
  - Componentes necessários para 80% de variância: {n_components_80}
  - Componentes necessários para 90% de variância: {n_components_90}
  - 3 primeiros componentes explicam: {pca_3.explained_variance_ratio_.sum()*100:.1f}% of variance

K-MEANS CLUSTERING:
  - Optimal K: {OPTIMAL_K} clusters
  - Silhouette Score: {silhouette_score(X_cluster, df['cluster']):.4f}
  - Cluster sizes: {dict(df['cluster'].value_counts().sort_index())}

KEY FINDINGS:
  - Municipalities can be grouped based on socioeconomic characteristics
  - Population size and income are major differentiating factors
  - Regional patterns are visible in cluster composition
  - Growth trajectories (2010-2022) vary significantly across clusters
""")


In [ ]:
# Salva os dados agrupados
output_columns = ['codigo_municipio', 'nome_municipio', 'codigo_estado', 'nome_estado',
                  'codigo_regiao', 'nome_regiao', 'cluster', 'PC1', 'PC2', 'PC3'] + raw_features

df_output = df[output_columns].copy()
print(f"Dataset de saída pronto com {len(df_output)} linhas e {len(output_columns)} colunas")
df_output.head()


In [ ]:
# Salva no S3 (opcional)
# output_key = 'gold/clustered_municipalities/data.parquet'
# with tempfile.NamedTemporaryFile(suffix='.parquet') as tmp:
#     df_output.to_parquet(tmp.name, index=False)
#     aws_profile = os.getenv("AWS_PROFILE") or AWS_PROFILE
#     session = boto3.Session(profile_name=aws_profile)
#     s3 = session.client('s3')
#     s3.upload_file(tmp.name, BUCKET_NAME, output_key)
#     print(f"Saved to s3://{BUCKET_NAME}/{output_key}")


---

## Fim da Análise

Este notebook demonstrou:
1. Carregamento e validação do dataset consolidado de municípios
2. Estatísticas descritivas e distribuições de features
3. PCA para redução de dimensionalidade (identificando componentes-chave de variância)
4. Agrupamento K-means para agrupar municípios por características socioeconômicas
5. Perfil e interpretação dos clusters

**Próximos Passos:**
- Usar os rótulos de cluster para análise estratificada
- Investigar padrões de compliance dentro de cada cluster
- Comparar características dos clusters com dados de sanções

## 6. Síntese Entre Notebooks

Esta análise de agrupamento é o quarto e último notebook analítico do pipeline da tese. Abaixo um resumo de como as descobertas dos quatro notebooks se conectam.

### Cadeia de Evidências

| Notebook | Método | Principal Achado |
|----------|--------|------------------|
| **NB01 — EDA** | Estatísticas descritivas | Renda é o correlato mais forte de sanções por 100k (r = 0,74). O Distrito Federal é um outlier estrutural. A distribuição é fortemente assimétrica à direita. |
| **NB02 — Estatística** | Regressão OLS | R² = 0,835 — renda + dummies regionais explicam 83,5% da variância de sanções. Norte e Nordeste têm mais sanções do que o esperado após controlar por renda. A alfabetização perde significância quando a renda está presente. |
| **NB03 — Machine Learning** | ElasticNet, RF, Logística | Confirma a renda como preditor dominante em todos os métodos. Classificação fraca com n = 27 (apenas exploratória). |
| **NB04 — Agrupamento** | PCA + K-means | O Brasil tem estrutura municipal dual: Sul/Sudeste desenvolvidos vs Norte/Nordeste menos desenvolvidos. 3 PCs explicam 84,4% da variância. K = 4 clusters (silhueta = 0,288). |

### Argumento Convergente da Tese

As sanções de compliance público brasileiras **não estão distribuídas aleatoriamente** — elas estão fortemente associadas a indicadores de desenvolvimento socioeconômico, particularmente renda, e exibem padrões regionais persistentes que sobrevivem a controles estatísticos.

A explicação mais parcimoniosa é a **capacidade institucional de detecção**: jurisdições de maior renda possuem infraestrutura de auditoria mais robusta, produzindo mais sanções registradas independentemente dos níveis reais de irregularidades. Norte e Nordeste apresentam mais sanções do que o esperado após controlar por renda, sugerindo fatores de governança além do desenvolvimento socioeconômico.

O agrupamento em nível municipal confirma que a estrutura dual socioeconômica do Brasil (Sul/Sudeste desenvolvidos vs Norte/Nordeste menos desenvolvidos) molda tanto a geração de transferências públicas quanto a capacidade institucional de monitorá-las.

### Limitações

- **n = 27** no nível estadual limita o poder estatístico para modelos de regressão e ML
- **Design transversal** — não é possível estabelecer causalidade
- **Viés de detecção** — sanções medem violações registradas, não irregularidades reais
- **Distrito Federal** — outlier estrutural que influencia todos os modelos
- **Deflação de renda** — os resultados dependem da série de deflatores do IPCA para BRL de 2022

*Conclusão completa da tese: ver `docs/thesis_conclusion.md` / `docs/thesis_conclusion.pt-BR.md`*


## 7. Mapas do Brasil (Achados em Estilo QGIS no Notebook)

Esta seção renderiza os achados finais da tese em mapas do Brasil em duas visões:

1. **Por estado**: intensidade de sanções por 100k enriquecida com a composição de cluster das cidades.
2. **Pelas principais cidades**: top municípios por população, coloridos por cluster e anotados com métricas de compliance.

Os assets de mapa são gerados a partir dos datasets Gold atuais e das fronteiras oficiais do IBGE.
        


In [ ]:
# Carrega datasets adicionais para os mapas de achados finais.
# presentation_assets.* APIs require English column names. We therefore keep
# English copies (`state_findings_en`, `df_city_en`) for downstream API calls
# and translate to Portuguese (`state_findings`, `df_city`) for display code.
from src.analysis.local_data_loader import LocalGoldDataLoader as _EnLoader
from src.analysis.presentation_assets import build_state_final_findings
from src.config.pt_br_translations import translate_dataframe_columns

_en_loader = _EnLoader()
df_state_en = _en_loader.load_dataset('analysis_compliance')
df_city_en = _en_loader.load_dataset('analysis_compliance_municipality')

cluster_assignments_en = df[['codigo_municipio', 'cluster']].rename(
    columns={'codigo_municipio': 'municipality_code'}
).copy()
cluster_assignments_en['municipality_code'] = (
    cluster_assignments_en['municipality_code'].astype(str).str.zfill(7)
)

df_state_en['state_code'] = df_state_en['state_code'].astype(str).str.zfill(2)
df_city_en['municipality_code'] = df_city_en['municipality_code'].astype(str).str.zfill(7)
df_city_en['state_code'] = df_city_en['state_code'].astype(str).str.zfill(2)

state_findings_en, cluster_state_mix_en = build_state_final_findings(
    state_df=df_state_en,
    city_df=df_city_en,
    cluster_assignments=cluster_assignments_en,
)

# Portuguese-labelled versions for display (choropleth hover, tooltips).
df_state = translate_dataframe_columns(df_state_en)
df_city = translate_dataframe_columns(df_city_en)
state_findings = translate_dataframe_columns(state_findings_en)
cluster_state_mix = translate_dataframe_columns(cluster_state_mix_en)
cluster_assignments = cluster_assignments_en.rename(
    columns={'municipality_code': 'codigo_municipio'}
)

city_with_cluster = df_city.merge(cluster_assignments, on='codigo_municipio', how='left')

print(f'State findings rows: {len(state_findings)}')
print(f'City findings rows: {len(city_with_cluster):,}')
state_findings.head()


In [ ]:
# Constrói / carrega o GeoJSON dos estados a partir das fronteiras oficiais do IBGE
from pathlib import Path
import zipfile
import tempfile

try:
    import shapefile  # pyshp
except ImportError as exc:
    raise ImportError(
        "pyshp is required for map rendering. Install with: pip install pyshp>=2.3.1"
    ) from exc

from src.analysis.presentation_assets import download_ibge_boundary_zip, build_state_geojson_from_shapefile

map_assets_dir = Path("..") / "docs" / "thesis_presentation_assets" / "qgis"
map_assets_dir.mkdir(parents=True, exist_ok=True)

state_zip_path = map_assets_dir / "BR_UF_2022.zip"
if not state_zip_path.exists():
    state_zip_path = download_ibge_boundary_zip("BR_UF_2022.zip", output_dir=map_assets_dir)

state_geojson_path = map_assets_dir / "brazil_states_final_findings.geojson"
build_state_geojson_from_shapefile(
    state_zip_path=state_zip_path,
    state_findings=state_findings_en,
    output_geojson_path=state_geojson_path,
)

with open(state_geojson_path, "r", encoding="utf-8") as _f:
    state_geojson = _json.load(_f)

print(f"State GeoJSON ready: {state_geojson_path}")
        


In [ ]:
# Mapa 1: Brasil por estado (sanções por 100k + contexto de cluster)
fig_state = px.choropleth_mapbox(
    state_findings,
    geojson=state_geojson,
    locations="codigo_estado",
    featureidkey="properties.state_code",
    color="sanctions_per_100k_state",
    hover_name="nome_estado",
    hover_data={
        "nome_regiao": True,
        "dominant_cluster": True,
        "dominant_cluster_share_pct": ":.1f",
        "avg_city_sanctions_per_100k": ":.2f",
        "n_cities": True,
    },
    color_continuous_scale="YlOrRd",
    mapbox_style="carto-positron",
    zoom=3.2,
    center={"lat": -14.2, "lon": -52.9},
    opacity=0.78,
    title="Brazil by State: Sanctions per 100k with City-Cluster Enrichment",
)
fig_state.update_layout(margin={"r": 0, "t": 60, "l": 0, "b": 0})
fig_state.show()
        


In [ ]:
# Constrói pontos centroides dos polígonos oficiais de municípios (para o mapa de cidades principais)
municipality_zip_path = map_assets_dir / "BR_Municipios_2022.zip"
if not municipality_zip_path.exists():
    municipality_zip_path = download_ibge_boundary_zip("BR_Municipios_2022.zip", output_dir=map_assets_dir)

with tempfile.TemporaryDirectory(prefix="ibge_muni_shape_") as _tmp_dir:
    with zipfile.ZipFile(municipality_zip_path) as _zip_file:
        _zip_file.extractall(_tmp_dir)

    _shp_path = next(Path(_tmp_dir).glob("*.shp"))
    _reader = shapefile.Reader(str(_shp_path))
    _fields = [field[0] for field in _reader.fields[1:]]
    _code_idx = _fields.index("CD_MUN")

    point_rows = []
    for _shape_record in _reader.iterShapeRecords():
        municipality_code = str(_shape_record.record[_code_idx]).zfill(7)
        xmin, ymin, xmax, ymax = _shape_record.shape.bbox
        point_rows.append(
            {
                "codigo_municipio": municipality_code,
                "lon": (xmin + xmax) / 2.0,
                "lat": (ymin + ymax) / 2.0,
            }
        )
    
    _reader.close()  # Close before temp cleanup (Windows fix)

municipality_points = pd.DataFrame(point_rows)
print(f"Municipality centroid points loaded: {len(municipality_points):,}")
municipality_points.head()
        


In [ ]:
# Mapa 2: Principais cidades (top por população) com cluster e achados de compliance
MAIN_CITIES_N = 120

main_cities = (
    city_with_cluster.sort_values("populacao_2022", ascending=False)
    .head(MAIN_CITIES_N)
    .merge(municipality_points, on="codigo_municipio", how="left")
    .copy()
)

main_cities["cluster_label"] = main_cities["cluster"].apply(
    lambda x: f"Cluster {int(x)}" if pd.notna(x) else "Not clustered"
)

missing_points = int(main_cities["lat"].isna().sum())
if missing_points > 0:
    print(f"Warning: {missing_points} main cities without centroid points; they will be excluded from map.")

main_cities_map = main_cities.dropna(subset=["lat", "lon"]).copy()
main_cities_map["population_2022_plot"] = pd.to_numeric(
    main_cities_map["populacao_2022"], errors="coerce"
).astype(float)

fig_main_cities = px.scatter_mapbox(
    main_cities_map,
    lat="lat",
    lon="lon",
    color="cluster_label",
    size="population_2022_plot",
    size_max=24,
    hover_name="nome_municipio",
    hover_data={
        "nome_estado": True,
        "populacao_2022": ":,.0f",
        "sancoes_por_100k": ":.2f",
        "renda_media_real_2022_brl_2022": ":.0f",
        "transferencia_media_per_capita": ":.2f",
        "cluster_label": False,
    },
    mapbox_style="carto-positron",
    zoom=3.2,
    center={"lat": -14.2, "lon": -52.9},
    title=f"Brazil Main Cities (Top {MAIN_CITIES_N} by Population): Cluster + Compliance Metrics",
)
fig_main_cities.update_layout(margin={"r": 0, "t": 60, "l": 0, "b": 0})
fig_main_cities.show()

main_cities_map[
    [
        "nome_municipio",
        "nome_estado",
        "populacao_2022",
        "cluster_label",
        "sancoes_por_100k",
        "renda_media_real_2022_brl_2022",
        "transferencia_media_per_capita",
    ]
].head(20)
        
